# LA Studio Translation — M2M-100 418M

This notebook loads exactly `m2m100-418m` (`facebook/m2m100_418M`) on CUDA.
It is independent from API Gateway and rejects every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into the matching LA Studio feature.


In [ ]:
!nvidia-smi
%pip install -q "fastapi==0.115.12" "uvicorn==0.34.3" "transformers==4.57.3" "accelerate==1.12.0" "sentencepiece==0.2.1" "safetensors==0.6.2"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_translation_worker.py')
WORKER.write_text('import os\nimport secrets\nimport threading\n\nimport torch\nfrom fastapi import Depends, FastAPI, Header, HTTPException\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.")\n\nfrom transformers import M2M100ForConditionalGeneration, M2M100Tokenizer\n\nMODEL_ID = "m2m100-418m"\nMODEL_NAME = "M2M-100 418M"\nUPSTREAM_MODEL = "facebook/m2m100_418M"\nUPSTREAM_REVISION = "55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636"\nSUPPORTED_LANGUAGES = "101 M2M100 language codes"\n\nTOKENIZER = M2M100Tokenizer.from_pretrained(UPSTREAM_MODEL, revision=UPSTREAM_REVISION)\nMODEL = M2M100ForConditionalGeneration.from_pretrained(\n    UPSTREAM_MODEL,\n    revision=UPSTREAM_REVISION,\n    torch_dtype=torch.float16,\n    low_cpu_mem_usage=True,\n).to("cuda").eval()\n\ndef translate_exact(texts: list[str], source: str, target: str) -> list[str]:\n    source = source.lower()\n    target = target.lower()\n    try:\n        TOKENIZER.src_lang = source\n        target_id = TOKENIZER.get_lang_id(target)\n    except KeyError as error:\n        raise HTTPException(status_code=422, detail=f"unsupported M2M100 language pair: {source} -> {target}") from error\n    inputs = TOKENIZER(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")\n    with torch.inference_mode():\n        output = MODEL.generate(**inputs, forced_bos_token_id=target_id, max_new_tokens=512)\n    return TOKENIZER.batch_decode(output, skip_special_tokens=True)\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TRANSLATION_TOKEN"]\nMAX_TRANSLATION_SEGMENTS = 128\nMAX_TRANSLATION_CHARS = 50000\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(authorization, "Bearer " + TOKEN):\n        raise HTTPException(status_code=401, detail="invalid or missing bearer token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\nclass TranslationSegment(BaseModel):\n    id: str = Field(min_length=1, max_length=128)\n    sourceText: str = Field(min_length=1, max_length=5000)\n\nclass TranslationRequest(BaseModel):\n    model: str\n    source_language: str = Field(min_length=2, max_length=12)\n    target_language: str = Field(min_length=2, max_length=12)\n    segments: list[TranslationSegment]\n\napp = FastAPI(title=f"LA Studio Translation - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(_: None = Depends(authorize)):\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(_: None = Depends(authorize)):\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "translation",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/translations")\ndef translate(request: TranslationRequest, _: None = Depends(authorize)):\n    require_exact_model(request.model)\n    if not request.segments:\n        raise HTTPException(status_code=400, detail="segments must not be empty")\n    texts = [item.sourceText.strip() for item in request.segments]\n    if any(not text for text in texts):\n        raise HTTPException(status_code=400, detail="each segment needs sourceText")\n    if len(texts) > MAX_TRANSLATION_SEGMENTS or sum(map(len, texts)) > MAX_TRANSLATION_CHARS:\n        raise HTTPException(status_code=413, detail="translation request is too large")\n    if not INFERENCE_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="worker is busy; retry shortly")\n    try:\n        translated = translate_exact(\n            texts,\n            request.source_language.strip(),\n            request.target_language.strip(),\n        )\n        if len(translated) != len(request.segments):\n            raise RuntimeError("model returned a different number of translations")\n        return {\n            "patches": [\n                {"id": item.id, "targetText": text.strip(), "state": "translated"}\n                for item, text in zip(request.segments, translated)\n            ]\n        }\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} translation failed: {type(error).__name__}: {str(error)[:300]}",\n        ) from error\n    finally:\n        INFERENCE_SLOTS.release()' + '\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'm2m100-418m'

import json
import os
import secrets
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env['LA_STUDIO_COLAB_TRANSLATION_TOKEN'] = TOKEN
log_path = '/content/la_studio_translation_worker.log'
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", 'la_studio_translation_worker:app', "--host", "127.0.0.1", "--port", '3943'],
    cwd="/content",
    env=env,
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        request = urllib.request.Request(
            "http://127.0.0.1:3943/health",
            headers={"Authorization": "Bearer " + TOKEN},
        )
        with urllib.request.urlopen(request, timeout=5) as response:
            health = json.load(response)
        if health.get("ready") and health.get("device") == "cuda" and health.get("model") == MODEL_ID:
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Exact-model worker did not become ready. Log tail:\n" + Path(log_path).read_text(errors="replace")[-5000:])

subprocess.run(
    ["wget", "-q", "-O", "/content/cloudflared.deb",
     "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
    check=True,
)
subprocess.run(["dpkg", "-i", "/content/cloudflared.deb"], check=True)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3943", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(180):
    line = tunnel.stdout.readline()
    print(line, end="")
    if "https://" in line and "trycloudflare.com" in line:
        public_url = line[line.find("https://"):].split()[0]
        break
if not public_url:
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_TRANSLATION_URL=" + public_url)
print("LA_STUDIO_COLAB_TRANSLATION_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_TRANSLATION_MODEL=" + MODEL_ID)
print("Paste only these direct-worker values into the matching LA Studio feature. Do not add /v1.")
